# Predict Kaggle `test.csv` → `submission.csv`
1. Re-use the recipe from `train.ipynb`: same features, params, and the tree count chosen by 5-fold CV.
2. Refit on **all** labelled rows (the held-out score has already been measured).
3. Apply the same row-level feature engineering to Kaggle's `test.csv`, predict, and write the file.

In [1]:
%run train.ipynb

18 features: ['homeplanet', 'cryosleep', 'destination', 'vip', 'deck', 'side', 'age', 'group_size', 'solo', 'cabin_num', 'total_spend', 'no_spend', 'log_roomservice', 'log_foodcourt', 'log_shoppingmall', 'log_spa', 'log_vrdeck', 'log_total_spend']


(5215, 18) (1739, 18) (1739, 18)
(5215,) (1739,) (1739,)


fold 0: acc=0.8109  trees=973


fold 1: acc=0.8167  trees=901


fold 2: acc=0.8160  trees=1021


fold 3: acc=0.8095  trees=1138


fold 4: acc=0.8115  trees=806
CV accuracy = 0.8129 ± 0.0029  | trees for final model: 973
0:	learn: 0.7626079	test: 0.7492812	best: 0.7492812 (0)	total: 39.5ms	remaining: 1m 58s


200:	learn: 0.8214765	test: 0.7987349	best: 0.8010351 (185)	total: 9.75s	remaining: 2m 15s


400:	learn: 0.8448706	test: 0.8044853	best: 0.8056354 (393)	total: 19.6s	remaining: 2m 7s


600:	learn: 0.8671141	test: 0.8056354	best: 0.8073606 (543)	total: 29.6s	remaining: 1m 58s


Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.807360552
bestIteration = 543

Shrink model to first 544 iterations.
train      acc=0.8531  f1=0.8552  auc=0.9416
validation acc=0.8074  f1=0.8078  auc=0.8964
test       acc=0.8051  f1=0.8066  auc=0.9099


## Refit on all labelled data

In [2]:
X_full = to_model_input(df_fe)
y_full = df_fe[target].astype(int)

final_model = CatBoostClassifier(**CATBOOST_PARAMS, iterations=n_trees, verbose=0)
final_model.fit(X_full, y_full, cat_features=CAT_FEATURES)

print("Trees:", n_trees, "| rows:", len(X_full), "| features:", len(selected_features))

Trees: 973 | rows: 8693 | features: 18


## Engineer Kaggle test features and predict
`engineer` uses only each row and its own group, so it learns nothing from the test set.

In [3]:
kaggle_test = standardize_columns(pd.read_csv("./data/raw/test.csv"))
X_kaggle = to_model_input(engineer(kaggle_test))

kaggle_pred = final_model.predict(X_kaggle).astype(int).ravel()
print(X_kaggle.shape)
print("Predicted transported rate:", kaggle_pred.mean().round(4),
      "| train rate:", y_full.mean().round(4))

(4277, 18)
Predicted transported rate: 0.5144 | train rate: 0.5036


## Write `submission.csv`
Kaggle wants `PassengerId,Transported` with `True`/`False`, one row per test passenger.

In [4]:
submission = pd.DataFrame({
    "PassengerId": kaggle_test["passengerid"],
    "Transported": kaggle_pred.astype(bool),
})

sample = pd.read_csv("./data/raw/sample_submission.csv")
assert list(submission.columns) == list(sample.columns)
assert len(submission) == len(sample)
assert submission["PassengerId"].equals(sample["PassengerId"])
assert submission["Transported"].notna().all()

submission.to_csv("./submission.csv", index=False)
print("Saved ./submission.csv", submission.shape)
submission.head()

Saved ./submission.csv (4277, 2)


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True
